In [1]:
from abiasales.data_generation import *
from abiasales.cleaning import *
from abiasales.weights import *
from abiasales.stats import *
from abiasales.stratification import *
from abiasales.metrics import *
from abiasales.inference import *
from abiasales.power import *
from abiasales.results import *
from abiasales.simulation import *

In [2]:
np.random.seed(42)  # synthetic data and simulations are reproducible
pd.options.display.float_format = '{:,.2f}'.format

# DATA

In [3]:
df_a = create_test_data(uplift=0, n_users=1000000)
df_a['exp_group'] = 'A'
df_b = create_test_data(uplift=0.1, n_users=1000000)
df_b['exp_group'] = 'B'

df = pd.concat([df_a, df_b]).reset_index(drop=True)
del df_a, df_b

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 12 columns):
 #   Column         Dtype  
---  ------         -----  
 0   uid            int64  
 1   platform       str    
 2   country        str    
 3   strata         str    
 4   views          int64  
 5   clicks         int64  
 6   payer_flag     int64  
 7   purchases      int64  
 8   views_cov      float64
 9   clicks_cov     float64
 10  purchases_cov  float64
 11  exp_group      str    
dtypes: float64(3), int64(5), str(4)
memory usage: 183.1 MB


In [5]:
STRATS_COLS = ['platform', 'country']
FEATURE_COLS = ['views_cov', 'clicks_cov', 'purchases_cov']

# CLEANING

Handling outliers

Initial sample size

In [6]:
len(df)

2000000

Mean metrics which we will use for handling outliers

In [7]:
df.groupby('exp_group')[FEATURE_COLS].mean()

,views_cov,clicks_cov,purchases_cov
exp_group,,,
A,13.04,1.71,2.07
B,13.04,1.72,2.07


Removing outliers

In [8]:
df = remove_outliers(df, metrics=FEATURE_COLS, contamination=0.001)

Means became a little bit less

In [9]:
df.groupby('exp_group')[FEATURE_COLS].mean()

,views_cov,clicks_cov,purchases_cov
exp_group,,,
A,13.03,1.71,2.07
B,13.02,1.71,2.07


Checking that exactly 0.1% of samples were removed

In [10]:
len(df)

1998000

Capping values of metric

In [11]:
metrics_to_cap = ['purchases']
metrics_capped = [col+'_capped' for col in metrics_to_cap]

In [12]:
df[metrics_capped] = cap_above_quantile(df, metrics=metrics_to_cap, q=0.999)

In [13]:
print("Initial mean = \t\t", df[metrics_to_cap].mean().values[0])
print("Mean after capping = \t", df[metrics_capped].mean().values[0])

Initial mean = 		 2.173153153153153
Mean after capping = 	 2.1709784784784785


<br><br>Function help:

In [14]:
remove_outliers?

<br><br>Function help with the code:

In [15]:
remove_outliers??

# WEIGHTS

Calculation of Weights for Ratio Metrics
There are four methods in total:
- **uniform** – all weights are equal to 1. Essentially, this corresponds to computing the metric at the user level and then taking the average of per-user metrics.
- **size** – all weights are equal to the denominator values. This is equivalent to summing the numerator across all users and dividing by the sum of denominators across all users.
- **sqrt** – weights are equal to the square root of the denominator.
- **intra_corr** – intra-user correlation–aware weighting, which accounts for within-user correlation. See https://arxiv.org/abs/1911.03553.

In [16]:
calc_weights(df['clicks'], df['views'], weight_method='uniform')

array([1., 1., 1., ..., 1., 1., 1.], shape=(1998000,))

In [17]:
calc_weights(df['clicks'], df['views'], weight_method='size')

array([ 6,  1,  4, ...,  1,  2, 60], shape=(1998000,))

In [18]:
calc_weights(df['clicks'], df['views'], weight_method='sqrt')

array([2.44948974, 1.        , 2.        , ..., 1.        , 1.41421356,
       7.74596669], shape=(1998000,))

In [19]:
calc_weights(df['clicks'], df['views'], weight_method='intra_corr')

array([ 5.79954185,  1.        ,  3.91873067, ...,  1.        ,
        1.98626913, 42.61784759], shape=(1998000,))

# STATS

Functions for calculating the mean, standard deviation, confidence interval using the delta method, and the linearization coefficient.

In [20]:
df['ratios'] = df['clicks'] / df['views']
df['weight'] = calc_weights(df['clicks'], df['views'], weight_method='intra_corr')

Weighted mean

In [21]:
weighted_mean(df['ratios'], df['weight'])

np.float64(0.13776902917892145)

Weighted standard deviation

In [22]:
weighted_std(df['ratios'], df['weight'])

np.float64(0.1099475708053091)

Standard deviation of the weighted mean scaled to one observation: `std / sqrt(n)` is the standard error of the weighted mean. Unlike the weighted standard deviation, it accounts for the effective sample size reduced by unequal weights, so it is used in statistical tests.

In [23]:
weighted_mean_std(df['ratios'], df['weight'])

np.float64(0.1112426285368445)

General function for calculating the standard deviation

In [24]:
# Without weights
calc_std(num=df['clicks'], den=df['views'], weights=None, use_delta_method=False)

np.float64(0.23182099990080637)

In [25]:
# With weights
calc_std(num=df['clicks'], den=df['views'], weights=df['weight'], use_delta_method=False)

np.float64(0.1112426285368445)

In [26]:
# Using Delta-method
calc_std(num=df['clicks'], den=df['views'], use_delta_method=True)

np.float64(0.11376737444645738)

Compute the confidence interval of a relative effect using the delta method. The interval is symmetric around the observed uplift.

In [27]:
calc_delta_method_ratio_confint(
    mean_1=0.50, std_1=2, nobs_1=100000,
    mean_2=0.55, std_2=2, nobs_2=120000,
    alpha=0.05
)

(np.float64(0.06456130734476426), np.float64(0.13543869265523592))

Fieller's confidence interval of a relative effect. It is used by the z-test and the t-test for relative uplift: the interval excludes 0 exactly when the p-value is significant, and it is asymmetric like the distribution of a ratio. For large samples it is close to the delta method interval.

In [28]:
calc_fieller_ratio_confint(
    mean_1=0.50, std_1=2, nobs_1=100000,
    mean_2=0.55, std_2=2, nobs_2=120000,
    alpha=0.05
)

(np.float64(0.06522046935442671), np.float64(0.13613255576490246))

Linearization coefficient for transforming ratio metric into linear

In [29]:
calc_linearization_coef(num=df['clicks'], den=df['views'], weights=df['weight'])

np.float64(0.139259470871782)

# STRATIFICATION

Adding strata column (always named 'unified_strata')

In [30]:
add_strata_col(df=df, strats_cols=STRATS_COLS)

In [31]:
df['unified_strata'].head()

0     (mobile, RU)
1    (desktop, RU)
2    (mobile, INT)
3     (mobile, RU)
4     (mobile, RU)
Name: unified_strata, dtype: object

Calculating strata weights: shares of the unit weight in each stratum. By default every unit has weight 1, with `weights` (e.g. views for a ratio metric with size weights) the stratified estimate keeps the ratio of sums.

In [32]:
calc_strats_weights(df=df, strats_cols=STRATS_COLS)

{('desktop', 'INT'): 0.24953103103103103,
 ('desktop', 'RU'): 0.24994844844844843,
 ('mobile', 'INT'): 0.2501771771771772,
 ('mobile', 'RU'): 0.25034334334334335}

# METRICS

General functions for calculating the mean and standard deviation with stratification, CUPED, or linearization adjustments.

Calculation of the mean and standard deviation accounting for stratification and the delta method, using precomputed weights.

In [33]:
calc_mean_and_std(
    df=df,
    metric_num='clicks',
    metric_den='views',
    metric_weight='weight',
    use_delta_method=False,
    use_stratification=False,
    strats_cols=None,
    strats_weights=None
)

(np.float64(0.13776902917892145), np.float64(0.1112426285368445))

A complete pipeline for calculating the mean and standard deviation, accounting for:
- weight computation
- stratification
- Delta method
- linearization
- CUPED

In [34]:
calc_pipeline_mean_and_std(
    df=df,
    metric_num='clicks',
    metric_den='views',
    weight_method='intra_corr',
    apply_linearization=False,
    use_delta_method=False,
    use_stratification=False,
    strats_cols=None,
    var_reduction_method=None,
    var_reduction_covariates=None
)

(np.float64(0.13776902917892145), np.float64(0.1112426285368445))

In [35]:
# Size weighting and Delta method (the Delta method estimates the ratio of sums, so it requires 'size' weights)
calc_pipeline_mean_and_std(
    df=df,
    metric_num='clicks',
    metric_den='views',
    weight_method='size',
    apply_linearization=False,
    use_delta_method=True,
    use_stratification=False,
    strats_cols=None,
    var_reduction_method=None,
    var_reduction_covariates=None
)

(np.float64(0.13814183496907115), np.float64(0.11376737444645738))

A complete pipeline for calculating the mean and standard deviation, accounting for:
- weight computation
- stratification
- Delta method
- linearization
- CUPED

Returns not only mean and std, but also weights with calculated ratios.

In [36]:
mean_a, std_a, weights_a, ratios_a, mean_b, std_b, weights_b, ratios_b = calc_groups_stats(
    df_control=df[df['exp_group']=='A'],
    df_treatment=df[df['exp_group']=='B'],
    metric_num='clicks',
    metric_den='views',
    weight_method='size',
    apply_linearization=False,
    use_delta_method=True,
    use_stratification=False,
    strats_cols=None,
    var_reduction_method=None,
    var_reduction_covariates=None
)

In [37]:
mean_a, std_a, mean_b, std_b

(np.float64(0.1315401881535686),
 np.float64(0.10964761901755046),
 np.float64(0.14474019424740667),
 np.float64(0.1161648068735179))

# INFERENCE

Check that the proportion of the control group does not deviate significantly from the expected level.

In [38]:
check_groups_distribution(df=df, exp_group_col='exp_group', control_name='A', control_perc=0.5)

,
Expected control group %,0.50
Fact control group %,0.50
Difference p-value,0.98


Score (Miettinen-Nurminen) test for proportions, returning the p-value and the confidence interval of the uplift. The interval contains the differences or ratios the test does not reject, so it agrees with the p-value.

In [39]:
calc_stattest_proportion_test(
    proportion_1=df.loc[df['exp_group']=='A', 'payer_flag'],
    proportion_2=df.loc[df['exp_group']=='B', 'payer_flag'],
    uplift_type='rel', # 'rel' – относительный аплифт, 'abs' – абсолютный аплифт
    confint_alpha=0.05 # значимость доверительного интервала
)

(np.float64(0.6049722693846753),
 (np.float64(-0.0060567159306159635), np.float64(0.003544329816417502)))

z-test, returning the p-value and the confidence interval of the uplift.

In [40]:
mean_a, std_a, _, _, mean_b, std_b, _, _ = calc_groups_stats(
    df_control=df[df['exp_group']=='A'],
    df_treatment=df[df['exp_group']=='B'],
    metric_num='payer_flag'
)

In [41]:
calc_stattest_z_test(
    mean_1=mean_a, # mean of the first group
    std_1=std_a, # standard deviation of the first group
    nobs_1=len(df.loc[df['exp_group']=='A', 'payer_flag']), # number of observations in the first group
    mean_2=mean_b, # mean of the second group
    std_2=std_b, # standard deviation of the second group
    nobs_2=len(df.loc[df['exp_group']=='B', 'payer_flag']), # number of observations in the second group
    uplift_type='rel', # 'rel' – relative uplift, 'abs' –absolute uplift
    confint_alpha=0.05 # significance level of confidence interval
)

(np.float64(0.6049723381598722),
 (np.float64(-0.006056751152530904), np.float64(0.0035443493030034023)))

t-test, returning the p-value and the confidence interval of the uplift.

In [42]:
calc_stattest_t_test(
    mean_1=mean_a, # mean of the first group
    std_1=std_a, # standard deviation of the first group
    nobs_1=len(df.loc[df['exp_group']=='A', 'payer_flag']), # number of observations in the first group
    mean_2=mean_b, # mean of the second group
    std_2=std_b, # standard deviation of the second group
    nobs_2=len(df.loc[df['exp_group']=='B', 'payer_flag']), # number of observations in the second group
    uplift_type='rel', # 'rel' – relative uplift, 'abs' –absolute uplift
    confint_alpha=0.05 # significance level of confidence interval
)

(np.float64(0.604972395421747),
 (np.float64(-0.0060567540467360015), np.float64(0.0035443522251419335)))

Poisson bootstrap, returning the p-value and the confidence interval of the uplift. The p-value comes from a symmetric test of the difference of means, the interval inverts this test, so it agrees with the p-value; for relative uplift it is asymmetric like Fieller's interval.

In [43]:
calc_stattest_bootstrap(
    metric_1=df[df['exp_group']=='A'].sample(100000).reset_index()['payer_flag'],
    weights_1=None,
    metric_2=df[df['exp_group']=='B'].sample(100000).reset_index()['payer_flag'],
    weights_2=None,
    uplift_type='rel', # 'rel' – relative uplift, 'abs' –absolute uplift
    confint_alpha=0.05, # significance level of confidence interval
    n_bootstrap=500 # number of bootstrap iterations
)

(np.float64(0.10778443113772455),
 (np.float64(-0.002202000076930055), np.float64(0.027645379503497436)))

General wrapper for all statistical tests.

In [44]:
calc_stattest(
    metric_1=df.loc[df['exp_group']=='A', 'payer_flag'],
    metric_2=df.loc[df['exp_group']=='B', 'payer_flag'],
    stat_method='proportion', uplift_type='rel', confint_alpha=0.05, n_bootstrap=500
)

,mean_1,mean_2,uplift_type,uplift,confint,pvalue
0,0.25,0.25,rel,-0.00,"(-0.006057, 0.003544)",0.60


In [45]:
calc_stattest(
    mean_1=mean_a, std_1=std_a, nobs_1=len(df.loc[df['exp_group']=='A', 'payer_flag']),
    mean_2=mean_b, std_2=std_b, nobs_2=len(df.loc[df['exp_group']=='B', 'payer_flag']),
    stat_method='t_test', uplift_type='rel', confint_alpha=0.05
)

,mean_1,mean_2,uplift_type,uplift,confint,pvalue
0,0.25,0.25,rel,-0.00,"(-0.006057, 0.003544)",0.60


In [46]:
calc_stattest(
    mean_1=mean_a, std_1=std_a, nobs_1=len(df.loc[df['exp_group']=='A', 'payer_flag']),
    mean_2=mean_b, std_2=std_b, nobs_2=len(df.loc[df['exp_group']=='B', 'payer_flag']),
    stat_method='z_test', uplift_type='rel', confint_alpha=0.05
)

,mean_1,mean_2,uplift_type,uplift,confint,pvalue
0,0.25,0.25,rel,-0.00,"(-0.006057, 0.003544)",0.60


In [47]:
calc_stattest(
    metric_1=df[df['exp_group']=='A'].sample(100000).reset_index()['payer_flag'],
    metric_2=df[df['exp_group']=='B'].sample(100000).reset_index()['payer_flag'],
    stat_method='bootstrap', uplift_type='rel', confint_alpha=0.05, n_bootstrap=500
)

,mean_1,mean_2,uplift_type,uplift,confint,pvalue
0,0.25,0.25,rel,-0.01,"(-0.020077, 0.009699)",0.47


# POWER

Sample size calculation accounting for stratification and CUPED. By default the treatment is assumed not to change the std of the metric; `effect='multiplicative'` scales the treatment std by (1 + uplift) and gives larger samples, which matters for large MDEs.

In [48]:
table, (sample_size_base,) = calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5,
    return_power_list=True
)
table

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,30374


In [49]:
# WITH STRATIFICATION
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    use_stratification=True,
    strats_cols=STRATS_COLS,
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,26741


In [50]:
# WITH CUPED
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    var_reduction_method='cuped',
    var_reduction_covariates=FEATURE_COLS,
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,27672


In [51]:
# WITH STRATIFICATION AND CUPED
table, (sample_size_strat_cuped,) = calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    use_stratification=True,
    strats_cols=STRATS_COLS,
    var_reduction_method='cuped',
    var_reduction_covariates=FEATURE_COLS,
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5,
    return_power_list=True
)
table

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,23086


MDE calculation for the sample sizes computed above: without anything and with stratification and CUPED

In [52]:
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='mde',
    alpha=[0.05], power=[0.8], sample_size=[sample_size_base, sample_size_strat_cuped], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
23086,5.7%
30374,5.0%


In [53]:
# WITH STRATIFICATION AND CUPED
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='mde',
    use_stratification=True,
    strats_cols=STRATS_COLS,
    var_reduction_method='cuped',
    var_reduction_covariates=FEATURE_COLS,
    alpha=[0.05], power=[0.8], sample_size=[sample_size_base, sample_size_strat_cuped], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
23086,5.0%
30374,4.4%


# RESULTS

In [54]:
df_a = create_test_data(uplift=0, n_users=30000)
df_a['exp_group'] = 'A'
df_b = create_test_data(uplift=0.05, n_users=30000)
df_b['exp_group'] = 'B'

df_results = pd.concat([df_a, df_b]).reset_index(drop=True)
del df_a, df_b

List of metric config parameters:
- **num** - name of the column containing the numerator of the metric.
- **den** - name of the column containing the denominator of the metric. Defaults to None and can be omitted.
- **weight_method** - type of user weighting. Default is 'uniform'. Possible values: 'uniform', 'size', 'sqrt', 'intra_corr'.
- **apply_linearization** - whether to linearize a ratio metric. Default is False. The linearized metric is expressed in ratio units of its group, so means and uplifts equal those of the ratio of sums. Requires weight_method='uniform'.
- **use_delta_method** - whether to use the delta method for variance estimation (relevant only for ratio metrics). Requires weight_method='size', because the delta method estimates the ratio of sums.
- **stat_method** - statistical test used to calculate the p-value. Default is 't_test' (Welch's t-test). Possible values: 'z_test', 't_test', 'proportion' (for binary 0/1 metrics only), 'bootstrap'.
- **uplift_type** - 'rel' (relative, default) or 'abs' (absolute) uplift.
- **var_reduction_method** - variance reduction method. Default is None. Possible values: None, 'cuped', 'cupac'.
- **var_reduction_covariates** - list of covariates for CUPED/CUPAC. Required if var_reduction_method is set.
- **var_reduction_params** - parameters of the CatBoost model for CUPAC, e.g. {'iterations': 100}. Defaults to None.
- **handle_outliers** - how to handle outliers. Default is None. Possible values: remove (IsolationForest), cap (capping values). Outliers are found on all compared groups together. A binary metric cannot be capped, and capping raises an error if the quantile equals the minimum of the metric (e.g. rare purchases), because all non-zero values would be replaced by it.
- **outliers_contamination** - threshold to define an outlier. Default is 0.001. In case of handle_outliers="remove" – top outliers_contamination % of samples, in case of handle_outliers="cap" (capping values) – top quantile level of capping.
- **outliers_cols** – columns to detect outliers, relevant only for handle_outliers="remove". Defaults to the numerator and denominator columns. Pre-experiment columns are preferable: outliers of the metric itself are detected in both groups together, the treatment makes them more frequent in one group, and removing them biases the uplift towards zero.

Incompatible combinations raise a ValueError: e.g. the delta method without 'size' weights, linearization without 'uniform' weights, the proportion test with linearization, variance reduction, stratification or capping, the bootstrap with stratification.

Stratification is configured at the calc_exp_results function level and can be overridden in a metric config:
- **use_stratification** - whether to apply stratification. Defaults to False.
- **strats_cols** - list of columns used for stratification. Defaults to None. With use_stratification = True, CUPED/CUPAC models are also fitted per stratum. Stratum weights are shares of the unit weight (views for 'size' weights), so stratification does not change the metric. Strata with fewer than 10 expected units in the smallest group are merged into one with a warning.

The function returns two p-value values:
- **pvalue_init** – the initial p-value obtained from the statistical test.
- **pvalue** – the p-value adjusted for multiple hypothesis testing.

The returned table is numeric: **uplift** and **confint** are fractions for relative uplift (0.05 means +5%). The displayed table is formatted with `format_results(results)`.

P-values that cannot be computed (e.g. of a constant metric) are excluded from the correction with a warning. With more than two groups, all pairs are compared by default; `control_group` compares every group only with the control. `random_state` makes the bootstrap reproducible.

In [55]:
# Metrics
metrics = {
    'purchases': {
        'num': 'purchases',
        'weight_method': 'uniform',
        'stat_method': 't_test'
    },
    'CTR': {
        'num': 'clicks',
        'den': 'views',
        'weight_method': 'size',
        'stat_method': 't_test',
        'use_delta_method': True
    }
}

# Same metrics but with CUPED and outliers handling
metrics_cuped = {
    'purchases': {
        'num': 'purchases',
        'weight_method': 'uniform',
        'stat_method': 't_test',
        'var_reduction_method': 'cuped',
        'var_reduction_covariates': FEATURE_COLS,
        'handle_outliers': 'remove',
        'outliers_cols': FEATURE_COLS,
        'outliers_contamination': 0.005
    },
    'CTR': {
        'num': 'clicks',
        'den': 'views',
        'weight_method': 'size',
        'stat_method': 't_test',
        'use_delta_method': True,
        'var_reduction_method': 'cuped',
        'var_reduction_covariates': FEATURE_COLS,
        'handle_outliers': 'cap',
        'outliers_contamination': 0.01
    }
}

In [56]:
results = calc_exp_results(
    df_results, exp_group_col='exp_group', metrics=metrics
)

,metric,group_1,group_2,mean_1,mean_2,uplift_type,uplift,confint,pvalue_init,pvalue
0,purchases,A,B,2.073367,2.190600,rel,5.65%,"(1.98%, 9.47%)",0.002359,0.002359
1,CTR,A,B,0.131935,0.138020,rel,4.61%,"(3.25%, 6.00%)",0.000000,0.000000


In [57]:
# WITH STRATIFICATION
results = calc_exp_results(
    df_results, exp_group_col='exp_group', metrics=metrics,
    use_stratification=True, strats_cols=STRATS_COLS
)

,metric,group_1,group_2,mean_1,mean_2,uplift_type,uplift,confint,pvalue_init,pvalue
0,purchases,A,B,2.064866,2.199789,rel,6.53%,"(3.05%, 10.14%)",0.000191,0.000191
1,CTR,A,B,0.131996,0.137956,rel,4.52%,"(3.28%, 5.77%)",0.000000,0.000000


In [58]:
# WITH CUPED
results = calc_exp_results(
    df_results, exp_group_col='exp_group', metrics=metrics_cuped
)

,metric,group_1,group_2,mean_1,mean_2,uplift_type,uplift,confint,pvalue_init,pvalue
0,purchases,A,B,2.061920,2.157343,rel,4.63%,"(1.11%, 8.27%)",0.009507,0.009507
1,CTR,A,B,0.129987,0.135506,rel,4.25%,"(2.99%, 5.51%)",0.000000,0.000000


In [59]:
# WITH STRATIFICATION AND CUPED
results = calc_exp_results(
    df_results, exp_group_col='exp_group', metrics=metrics_cuped,
    use_stratification=True, strats_cols=STRATS_COLS
)

,metric,group_1,group_2,mean_1,mean_2,uplift_type,uplift,confint,pvalue_init,pvalue
0,purchases,A,B,2.053010,2.165931,rel,5.50%,"(2.26%, 8.85%)",0.000777,0.000777
1,CTR,A,B,0.130049,0.135271,rel,4.02%,"(2.85%, 5.20%)",0.000000,0.000000


# SIMULATION

Simulate:
- A/A tests to verify that the p-value properly controls the Type I error rate.
- A/B tests to compare the statistical power of different tests.

Each of the functions **`run_aa_test_simulation`** and **`run_ab_test_simulation`** takes a **`configs`** argument — a list of dictionaries, where each dictionary defines the metric configuration and the statistical methods to be used.  

Each dictionary has the following **structure**:

---

- **`num`** *(str)*  
  Name of the column with the metric’s numerator (e.g., `"purchases"`).

- **`den`** *(str | None)*  
  Name of the column with the metric’s denominator.  
  If `None`, assumes a value of 1.

- **`weight_method`** *(str)*  
  Unit-level weighting scheme to apply.  
  Allowed:  
  - `'uniform'`: all users weight = 1  
  - `'size'`: weights equal to the metric “size” (e.g., denominator or exposure)  
  - `'sqrt'`: weights are √(size)  
  - `'intra_corr'`: correlation-aware weights (accounts for within-user correlation)  
  Default: `'uniform'`.

- **`apply_linearization`** *(bool)*  
  Whether to linearize a ratio metric. Requires `weight_method='uniform'`.  
  Default: `False`.

- **`use_delta_method`** *(bool)*  
  Whether to use the delta method for a ratio metric. Requires `weight_method='size'`.  
  Default: `False`.

- **`stat_method`** *(str)*  
  Statistical test to compute the p-value.  
  Allowed: `'proportion'`, `'z_test'`, `'t_test'`, `'bootstrap'`.  
  Use `'proportion'` only for binary (0/1) metrics.  
  Default: `'t_test'`.

- **`use_stratification`** *(bool)*  
  Whether to apply stratification.  
  Default: `False`.

- **`strats_cols`** *(list[str] | None)*  
  Columns to use as strata keys (e.g., `device`, `country`).  
  With `use_stratification=True`, CUPED/CUPAC models are also fitted per stratum.  
  Default: `None`.

- **`var_reduction_method`** *(str)*  
  Variance reduction method.  
  Allowed: `None`, `'cuped'`, `'cupac'`.  
  Default: `None`.

- **`var_reduction_covariates`** *(list[str] | None)*  
  Covariates for CUPED/CUPAC.  
  Ignored if `var_reduction_method=None`.  
  Default: `None`.

- **`var_reduction_params`** *(dict | None)*  
  Parameters of the CatBoost model for CUPAC (e.g., `{'iterations': 100}`).  
  Default: `None`.

- **`handle_outliers`** *(str | None)*  
  Strategy for outliers.  
  Allowed:  
  - `None`  
  - `'remove'`: drop outliers using IsolationForest  
  - `'cap'`: cap metric at quantile threshold  
  Default: `None`.

- **`outliers_contamination`** *(float | None)*  
  Expected outlier fraction (e.g., `0.001` = 0.1%).  
  Used to set cutoffs for `'remove'`/`'cap'`.  
  Ignored if `handle_outliers=None`.  
  Default: `0.001`.

- **`outliers_cols`** *(list[str])*  
  Columns on which to detect/handle outliers (e.g., `exposure`, `spend`, `price`).  
  Ignored if `handle_outliers != 'remove'`. Defaults to the numerator and denominator columns.  
  Prefer pre-experiment columns: removing outliers of the metric itself biases the uplift towards zero.

- **`uplift_type`** *(str)*  
  `'rel'` or `'abs'`.  
  Default: `'rel'`.

Configs are usually built with `construct_exp_configs`, which skips incompatible combinations of methods.

---

**Example:**

```python
{
    'num': 'purchases',
    'weight_method': 'uniform',
    'stat_method': 'z_test',
    'use_stratification': False,
    'strats_cols': None,
    'var_reduction_method': None,
    'var_reduction_covariates': ['purchases_cov'],    # ignored because var_reduction_method=None
    'handle_outliers': None,
    'outliers_contamination': 0.001,                  # ignored because handle_outliers=None
    'outliers_cols': ['purchases_cov']                # ignored because handle_outliers=None
}
```

In [60]:
df_sim = df[df['exp_group']=='A']
n_exp = 1000
sample_size = 1000

Estimate the MDE (Minimum Detectable Effect) for a given sample size.

In [61]:
pt, pw = calc_power_table(
    df=df_sim,
    metric_num='purchases',
    mode='mde',
    alpha=[0.05], power=[0.8], sample_size=[sample_size], control_perc=0.5,
    return_power_list=True
)
display(pt)

model_uplift = pw[0]

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
1000,27.8%


Construct a set of statistical tests to be compared.

In [62]:
configs = construct_exp_configs(
    metric_num='purchases', # Numerator of metric
    metric_den=None, # Denominator of metric
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[False, True], # False, True
    strats_cols=STRATS_COLS,
    var_reduction_methods=[None, 'cuped'], # None, 'cuped', 'cupac'
    var_reduction_covariates=FEATURE_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [63]:
configs

[{'num': 'purchases',
  'den': None,
  'weight_method': 'uniform',
  'apply_linearization': False,
  'use_delta_method': False,
  'stat_method': 't_test',
  'use_stratification': False,
  'strats_cols': ['platform', 'country'],
  'var_reduction_method': None,
  'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'],
  'var_reduction_params': None,
  'handle_outliers': None,
  'outliers_contamination': None,
  'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov']},
 {'num': 'purchases',
  'den': None,
  'weight_method': 'uniform',
  'apply_linearization': False,
  'use_delta_method': False,
  'stat_method': 't_test',
  'use_stratification': False,
  'strats_cols': ['platform', 'country'],
  'var_reduction_method': 'cuped',
  'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'],
  'var_reduction_params': None,
  'handle_outliers': None,
  'outliers_contamination': None,
  'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov']},
 {'num'

Generate synthetic A/A tests. `fpr` is the share of experiments with p-value < alpha, `fpr_valid` is False if it is significantly above alpha (one-sided binomial test).

In [64]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:01<00:00, 598.70it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:03<00:00, 333.04it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:02<00:00, 363.05it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:05<00:00, 181.70it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,fpr,fpr_valid,fpr-pvalue_auc
0,t_test,False,nan,None,None,0.054000,True,0.516082
1,t_test,False,cuped,None,None,0.051000,True,0.509140
2,t_test,True,nan,None,None,0.049000,True,0.478978
3,t_test,True,cuped,None,None,0.055000,True,0.496475


All statistical tests are valid

Generate synthetic A/B tests

In [65]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='value',   # Metric type – 'proportion', 'value', 'ratio',
                           # Uplifts will be generated differently depending on it.
    uplift=model_uplift
)

Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:03<00:00, 286.75it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:04<00:00, 203.69it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:04<00:00, 214.45it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:07<00:00, 135.31it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,power
0,t_test,False,nan,None,None,0.790000
1,t_test,False,cuped,None,None,0.818000
2,t_test,True,nan,None,None,0.828000
3,t_test,True,cuped,None,None,0.890000


Stratification and CUPED both increase power, together they give the highest one: 0.89 vs 0.79 for the plain t-test.